In [ ]:
import os
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "12000"
import sys, os
sys.setrecursionlimit(100000)
os.environ["DEEPEVAL_DISABLE_RICH_DISPLAY"] = "YES"
os.environ["FORCE_COLOR"] = "0"
os.environ["NO_COLOR"] = "1"


 

In [ ]:
import re
import json
import pandas as pd
import threading
from typing import List, Tuple, Union
from pydantic import BaseModel as PydanticModel

from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase
from deepeval import evaluate
from deepeval.metrics import (
    ContextualRelevancyMetric,
    ContextualRecallMetric,
    ContextualPrecisionMetric,
)

from rag_agent import RAGAgent
from llm_local import get_local_llm

agent = RAGAgent(
    pdf_paths=[
        r'documents\Company_Policies.pdf',
        r'documents\Company_Profile.pdf',
        r'documents\Product_and_Pricing.pdf',
    ],
    embedding_model_path=r"E:\ds-team\huggingface_models\Old BKP\test_models\bge_m3",
    embedding_device="cuda",
    qdrant_path=os.path.join("qdrant_store"),
    collection_name="company_docs",
    # confident_us_proj_SUaHShP5/OUay0OSJbzecHT74TsL4KF9XrDjCCcDwVI=
    force_recreate=True,
)   

In [ ]:
import re, json
from typing import Tuple, Union
from pydantic import BaseModel as PydanticModel
from deepeval.models import DeepEvalBaseLLM

import threading

class AgentLLM(DeepEvalBaseLLM):
    _gpu_lock = threading.Lock()

    def load_model(self): return agent.llm

    def generate(self, prompt: str, schema=None) -> Union[str, PydanticModel]:
        with self._gpu_lock:
            if schema:
                try:
                    structured_llm = agent.llm.with_structured_output(schema)
                    return structured_llm.invoke(prompt)
                except Exception as e:
                    print(f"[structured output failed, falling back to raw text]")
                    
            text = agent.llm.invoke(prompt).content
            
            if schema:
                return self._parse_json(text, schema)
                
            return text


    def _parse_json(self, text: str, schema) -> PydanticModel:
        # 1. Standard extraction
        for pattern in [r'```json\s*(.*?)\s*```', r'```\s*(.*?)\s*```', r'(\{.*\})']:
            # limit to 20,000 chars to avoid regex engine crashes
            m = re.search(pattern, text[:20000], re.DOTALL)
            if m:
                try:
                    return schema.model_validate(json.loads(m.group(1)))
                except Exception:
                    continue
                    
        # 2. BULLETPROOF EMERGENCY SALVAGE (No Regex)
        # We split the text manually to find complete JSON objects without crashing Python
        salvaged_objects = []
        
        # Split string on closing braces
        parts = text.split('}')
        for p in parts:
            if '"verdict"' in p:
                start = p.rfind('{')
                if start != -1:
                    json_str = p[start:] + '}'
                    try:
                        salvaged_objects.append(json.loads(json_str))
                    except json.JSONDecodeError:
                        pass
                        
        if salvaged_objects:
            print(f"[salvaged {len(salvaged_objects)} truncated verdicts]")
            root_key = list(schema.model_fields.keys())[0]
            return schema.model_validate({root_key: salvaged_objects})
            
        print(f"[WARN] Total parse failure. Raw output:\n{text[:400]}")
        raise ValueError(f"Could not extract valid JSON for schema '{schema.__name__}'")

    async def a_generate(self, prompt, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self): return "Qwen2.5-14B (local)"

llm = AgentLLM("Qwen2.5-14B-Instruct")

In [ ]:
import time
latency_map = {}
csv_path = "data/nexaai_golden_set_annotated.csv" 
try:
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} records from {csv_path}.")
except FileNotFoundError:
    print(f"Error: Could not find {csv_path}. Please make sure the CSV exists.")
    df = pd.DataFrame() 

test_cases = []

if not df.empty:
    print("Building test cases through the RAG pipeline...")
    for index, row in df.iterrows():
        user_query = str(row['input'])
        expected_ans = str(row['expected_output'])
        
        if 'context' in row and pd.notna(row['context']):
            ground_truth_context = [str(row['context'])] 
        else:
            ground_truth_context = []
    
        start_t = time.time()
        result = agent.invoke(user_query)
        latency = time.time() - start_t
        latency_map[user_query] = latency
        
        retrieved_docs = result.get('docs', [])
        retrieved_context = [d.page_content for d in retrieved_docs]
        actual_answer = result.get('answer', '')
    
        test_case = LLMTestCase(
            input=user_query,
            actual_output=actual_answer,
            expected_output=expected_ans,
            retrieval_context=retrieved_context,
            context=ground_truth_context  
        )
        
        test_cases.append(test_case)
    
    print(f"{len(test_cases)} test cases built ✓")


In [ ]:
import os
import json
import pandas as pd
from datetime import datetime

def save_eval_results(eval_result, output_dir="eval_results", run_tag=None):

    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S" )
    tag = run_tag or timestamp

    rows = []
    for tr in eval_result.test_results:
        base = {
            "test_case_name": tr.name,
            "input":          tr.input,
            "latency":        latency_map.get(tr.input, None),
            "actual_output":  tr.actual_output,
            "expected_output": tr.expected_output,
            "success":        tr.success,
        }
        for md in tr.metrics_data:
            row = base.copy()
            row.update({
                "metric":    md.name,
                "score":     md.score,
                "threshold": md.threshold,
                "passed":    md.success,
                "reason":    md.reason,
            })
            rows.append(row)

    df = pd.DataFrame(rows)

    # --- Save per-test-case CSV ---
    csv_path = os.path.join(output_dir, f"eval_{tag}.csv")
    df.to_csv(csv_path, index=False)
    print(f"✅ Detailed results saved → {csv_path}")

    # --- Save aggregate summary CSV ---
    summary_rows = []
    for metric_name in df["metric"].unique():
        sub = df[df["metric"] == metric_name]
        summary_rows.append({
            "metric":        metric_name,
            "avg_score":     round(sub["score"].mean(), 4),
            "pass_rate_%":   round(sub["passed"].mean() * 100, 1),
            "passed":        int(sub["passed"].sum()),
            "failed":        int((~sub["passed"]).sum()),
            "total":         len(sub),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(output_dir, f"summary_{tag}.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"✅ Summary saved       → {summary_path}")

    # --- Save full JSON (includes reasons & retrieval context) ---
    full_data = []
    for tr in eval_result.test_results:
        full_data.append({
            "name":            tr.name,
            "input":           tr.input,
            "latency":         latency_map.get(tr.input, None),
            "actual_output":   tr.actual_output,
            "expected_output": tr.expected_output,
            "success":         tr.success,
            "retrieval_context": tr.retrieval_context,
            "context":         tr.context,
            "metrics": [
                {
                    "metric":    md.name,
                    "score":     md.score,
                    "threshold": md.threshold,
                    "passed":    md.success,
                    "reason":    md.reason,
                }
                for md in tr.metrics_data
            ],
        })

    json_path = os.path.join(output_dir, f"eval_{tag}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"run_tag": tag, "results": full_data}, f, indent=2, ensure_ascii=False)
    print(f"✅ Full JSON saved      → {json_path}")

    # Display summary inline
    print("\n📊 Aggregate Summary:")
    print(summary_df.to_string(index=False))
    return df, summary_df


In [ ]:
relevancy = ContextualRelevancyMetric(model=llm, threshold=0.5, include_reason=False, async_mode=False)
recall    = ContextualRecallMetric(model=llm, threshold=0.5, include_reason=False, async_mode=False)
precision = ContextualPrecisionMetric(model=llm, threshold=0.5, include_reason=False, async_mode=False)

print("Metrics ready ✓")



In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

results = evaluate(
    test_cases=test_cases,
    metrics=[relevancy, recall, precision],
    
)
df_ctx, summary_ctx = save_eval_results(results, run_tag="contextual_metrics")

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import FaithfulnessMetric

answer_correctness = GEval(
    name="Answer Correctness",
    model=llm,             
    async_mode=False,     
    criteria="Evaluate if the actual output's 'answer' property is correct and complete from the input and retrieved context. If the answer is not correct or complete, reduce score.",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT]
)

faithfulness = FaithfulnessMetric(
    model=llm,
    threshold=0.5,
    include_reason=True,
    async_mode=False
)

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

generator_metrics = [answer_correctness,faithfulness]


results_geval = evaluate(
    test_cases=test_cases, 
    metrics=generator_metrics, 
)

df_geval, summary_geval = save_eval_results(results_geval, run_tag="geval_metrics")


In [ ]:


# from deepeval import evaluate

# generator_metrics = [answer_correctness, faithfulness]

# BATCH_SIZE = 10
# all_results = []

# for start in range(0, len(test_cases), BATCH_SIZE):
#     batch = test_cases[start : start + BATCH_SIZE]
#     print(f"Evaluating cases {start+1}–{min(start+BATCH_SIZE, len(test_cases))} / {len(test_cases)}...")
#     batch_result = evaluate(
#         test_cases=batch,
#         metrics=generator_metrics,
#     )
#     all_results.extend(batch_result.test_results)
#     print(f"  Done. ({len(all_results)} total evaluated)")

# results_geval = batch_result
# results_geval.test_results = all_results

# df_geval, summary_geval = save_eval_results(results_geval, run_tag="geval_metrics")

In [ ]:
# from deepeval import evaluate

# generator_metrics = [answer_correctness, faithfulness]

# results_geval = evaluate(test_cases, generator_metrics )

# df_geval, summary_geval = save_eval_results(results_geval, run_tag="geval_metrics")